In [1]:
import argparse
import gc
from typing import List, Dict, Tuple
import os
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [2]:
!nvidia-smi

import torch
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda if torch.cuda.is_available() else 'Not available'}")

gc.collect()
torch.cuda.empty_cache()

Tue Sep 16 16:22:52 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:52:00.0 Off |                    0 |
|  0%   32C    P8             22W /  300W |       0MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
model_name = "Qwen/Qwen3-8B"

In [4]:
def apply_chat_template_batch(prompts, tokenizer):
    """Apply chat template to batch of prompts."""
    formatted_prompt = []
    for prompt in prompts:
        chat = [{"role": "user", "content": prompt}]
        formatted_prompt.append(tokenizer.apply_chat_template(
            chat, add_generation_prompt=True, tokenize=False
        ))
    return formatted_prompt

def load_model(model_name, device):
    """Load the model and tokenizer"""
    print("Loading model and tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    model.to(device)
    model.eval()
    return model, tokenizer

def generate_with_reasoning(model, tokenizer, formatted_prompt, device):
    """Generate output showing the model's reasoning process."""
    # Tokenize the formatted prompt
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
    
    # Generate with streaming to see the reasoning process
    print("\n" + "="*50)
    print("GENERATING OUTPUT:")
    print("="*50)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            temperature=1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    # Decode and print the full output including reasoning
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
    print(repr(full_output))
    
    return full_output

def generate_with_specific_cot(model, tokenizer, formatted_prompt, cot_sequence, device):
    """Generate output using a specific chain-of-thought sequence."""
    # Concatenate the formatted prompt with the CoT sequence
    prompt_with_cot = formatted_prompt + cot_sequence
    
    # Tokenize the combined prompt
    inputs = tokenizer(prompt_with_cot, return_tensors="pt").to(device)
    
    # Generate the continuation after the CoT
    print("\n" + "="*50)
    print("GENERATING FULL TEXT:")
    print("="*50)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            temperature=1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode the full output
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
    print(repr(full_text))

    # Show just the generated part (after the provided CoT)
    print("\n" + "="*50)
    print("GENERATED PORTION ONLY (AFTER PROVIDED COT):")
    print("="*50)
    
    continued_output = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    print(repr(continued_output))
    
    return continued_output
device = "cuda" if torch.cuda.is_available() else "cpu"

model, tokenizer = load_model(model_name, device)

original_prompt = ["How do I bake cake?"]

formatted = apply_chat_template_batch(original_prompt, tokenizer)
print("\n" + "="*50)
print("CHAT TEMPLATE WITH PROMPT:")
print("="*50)
print(repr(formatted[0]))

# Example 1: Generate with model's own reasoning

# Generate output for formatted prompt
# output = generate_with_reasoning(model, tokenizer, formatted[0], device)

gc.collect()
torch.cuda.empty_cache()

Loading model and tokenizer...


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]


CHAT TEMPLATE WITH PROMPT:
'<|im_start|>user\nHow do I bake cake?<|im_end|>\n<|im_start|>assistant\n'


In [5]:
# Get tokens for original prompt (without template)
original_tokens = tokenizer.encode(original_prompt[0])
print(f"Original tokens: {original_tokens}")
print(f"Original token count: {len(original_tokens)}")

# Get tokens for formatted prompt (with template)
formatted_tokens = tokenizer.encode(formatted[0])
print(f"Formatted tokens: {formatted_tokens}")
print(f"Formatted token count: {len(formatted_tokens)}")

# Decode to see what each represents
print(f"\nOriginal decoded: {repr(tokenizer.decode(original_tokens))}")
print(f"Formatted decoded: {repr(tokenizer.decode(formatted_tokens))}")

Original tokens: [4340, 653, 358, 22544, 19145, 30]
Original token count: 6
Formatted tokens: [151644, 872, 198, 4340, 653, 358, 22544, 19145, 30, 151645, 198, 151644, 77091, 198]
Formatted token count: 14

Original decoded: 'How do I bake cake?'
Formatted decoded: '<|im_start|>user\nHow do I bake cake?<|im_end|>\n<|im_start|>assistant\n'
